# 🎬 FriendsOS — Colab Deployment

Deploy the FriendsOS multi-agent narrative engine on Google Colab with GPU.

### Prerequisites
- Colab with **GPU runtime** (T4 recommended)
- **ngrok** auth token → [https://ngrok.com](https://ngrok.com)

---
## Cell 1 — Verify GPU

In [14]:
!nvidia-smi

Sun Apr 19 20:45:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

---
## Cell 2 — Clone Repo & Install Deps
Replace `<YOUR_REPO_URL>` with your GitHub repo URL.

In [15]:
# !git clone <YOUR_REPO_URL> /content/friendsOS

!pip install -q python-dotenv fastapi uvicorn fakeredis redis requests pyngrok
print('\n✅ Dependencies installed')


✅ Dependencies installed


---
## Cell 3 — Install Ollama & Pull Model

In [16]:
# !curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, threading

def start_ollama():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

threading.Thread(target=start_ollama, daemon=True).start()
time.sleep(5)
print('Ollama server started, pulling model...')

!ollama pull llama3.1:8b
!ollama list
print('\n✅ Model ready')

Ollama server started, pulling model...

NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    

✅ Model ready


---
## Cell 4 — Quick Model Test

In [17]:
import requests, json

r = requests.post('http://localhost:11434/api/chat', json={
    'model': 'llama3.1:8b',
    'stream': False,
    'messages': [
        {'role': 'system', 'content': 'You are Chandler Bing from Friends. Respond fully in character.'},
        {'role': 'user', 'content': "How's your day going?"}
    ]
})

print('Chandler says:')
print(r.json()['message']['content'])

Chandler says:
Could it BE any better? I mean, I'm just sitting here, making jokes, and hanging out with my friends. What more could you ask for? Although, Monica's been driving me crazy all morning, nagging me about something or other... (smirk) You know, the usual.


---
## Cell 5 — Write `.env`

In [ ]:
env_content = """APP_ENV=production
USE_DUMMY_DATA=true
DIALOGUE_PROVIDER=ollama
CONVERGE_PROVIDER=ollama
OLLAMA_BASE_URL=http://localhost:11434
OLLAMA_DIALOGUE_MODEL=llama3.1:8b
OLLAMA_CONVERGE_MODEL=llama3.1:8b
REDIS_URL=redis://localhost:6379
REDIS_MODE=fakeredis
CHROMA_MODE=embedded
CHROMA_PERSIST_PATH=/content/chroma_data
CHROMA_COLLECTION_NAME=friendsos_memory
LANGCHAIN_TRACING_V2=false"""

with open('/content/friendsOS/backend/.env', 'w') as f:
    f.write(env_content)

print('✅ .env written (dummy_mode=true for safe testing)')

✅ .env written (dummy_mode=true for safe testing)


---
## Cell 6 — Start FastAPI + ngrok
Paste your **ngrok auth token** below.

In [ ]:
import subprocess, sys, time, threading, os, socket
from pyngrok import ngrok, conf

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NGROK_AUTHTOKEN = '3C2dneJf1eVGxCA6Ll1C6pKfyv9_5xGe4MvtzeYhd45e8o9Md'  # ← PASTE YOUR TOKEN
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

conf.get_default().auth_token = NGROK_AUTHTOKEN

# Add backend to Python path so imports work
backend_dir = '/content/friendsOS/backend'
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)

# Start uvicorn as a subprocess (not blocking)
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=backend_dir,
    env={**os.environ, 'PYTHONPATH': backend_dir},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Wait for server to actually start listening on port 8000
print('Starting FastAPI server...')
for i in range(30):
    time.sleep(1)
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', 8000))
        sock.close()
        if result == 0:
            print(f'  Server ready after {i+1}s')
            break
    except:
        pass
else:
    # Print server output for debugging
    out = server_proc.stdout.read(4096).decode() if server_proc.stdout else ''
    print(f'ERROR: Server failed to start in 30s. Output:\n{out}')
    raise RuntimeError('FastAPI server did not start')

# Now connect ngrok
public_url = ngrok.connect(8000, 'http')

print('=' * 60)
print(f'✅ FriendsOS backend is LIVE at:')
print(f'   {public_url}')
print('=' * 60)
print(f'\nOn your Mac, set frontend/.env to:')
print(f'  VITE_API_URL={public_url}')
print(f'  VITE_USE_DUMMY=false')
print(f'\nThen run: cd frontend && npm run dev')

Starting FastAPI server...
ERROR: Server failed to start in 30s. Output:
ERROR:    Error loading ASGI app. Could not import module "main".



RuntimeError: FastAPI server did not start

---
## Cell 7 — Verify the API is working

In [ ]:
import requests

base = 'http://localhost:8000'

print('Health:', requests.get(f'{base}/api/health').json())
print()
print('Agents:', [a['name'] for a in requests.get(f'{base}/api/agents/').json()])
print()
print('Episodes:', [e['title'] for e in requests.get(f'{base}/api/episodes/').json()])
print()
print('\n✅ All endpoints working!')

---
## Cell 8 — Keep Alive

In [ ]:
import time
print('🟢 Session active. Keep this tab open.')
i = 0
while True:
    time.sleep(60)
    i += 1
    print(f'  ⏱ alive {i}m', end='\r')